In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

from scipy.stats import linregress

In [ ]:
nav = pd.read_csv("../data/processed/02_nav_history.csv")

nav.head()
nav.info()
nav.dtypes
nav["date"] = pd.to_datetime(nav["date"])

In [ ]:
nav.dtypes
nav.head(10)
# If we use pct_change() directly, Python will calculate the return from: Last row of Fund A-First row of Fund B
# That is completely wrong. We need to calculate returns within each fund only.

# nav["daily_return"] = nav["nav"].pct_change()

# using group by function to calculate daily returns for each fund separately
nav.columns


In [ ]:
nav.groupby("amfi_code")
nav.sort_values(["amfi_code", "date"])
nav.head()
print(nav.columns.tolist())

In [ ]:
# Correct Version for Dataset
nav = nav.sort_values(["amfi_code", "date"])
# Calculate Daily Returns
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)
# Verify the Results
nav.head(10)
# Check the Statistics
nav["daily_return"].describe()

In [ ]:
# merging two datasets
fund = pd.read_csv("../data/processed/01_fund_master.csv")

fund.head()
fund.columns.tolist()
fund.info()
fund.columns

nav = nav.merge(
    fund[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "category",
            "expense_ratio_pct",
            "benchmark"
        ]
    ],
    on="amfi_code",
    how="left"
)

In [ ]:
# check the merged dataset
nav.head()

# Check row count
print(nav.shape)

# Check missing values
print(nav.isnull().sum())

# Check number of unique funds
print(nav["scheme_name"].nunique())


In [ ]:
# CAGR (Compound Annual Growth Rate).
# CAGR=(Ending Value/Beginning Value)^(1/n) - 1
# CAGR gives us a fair annualized comparison.
def calculate_cagr(nav_data, years):
    """
    Calculate CAGR for a given number of years.
    """

    trading_days = 252 * years

    # Check if enough data is available
    if len(nav_data) < trading_days:
        return np.nan

    # Sort by date
    nav_data = nav_data.sort_values("date")

    # Starting NAV
    start_nav = nav_data.iloc[-trading_days]["nav"]

    # Ending NAV
    end_nav = nav_data.iloc[-1]["nav"]

    # CAGR Formula
    cagr = (end_nav / start_nav) ** (1 / years) - 1

    return cagr




In [ ]:
first_fund = nav["scheme_name"].unique()[0]
fund_data = nav[
    nav["scheme_name"] == first_fund
]
calculate_cagr(fund_data, 1)
cagr_results = []
for scheme, group in nav.groupby("scheme_name"):
   cagr_results = []

for scheme, group in nav.groupby("scheme_name"):

    group = group.sort_values("date")

    cagr_1 = calculate_cagr(group, 1)
    cagr_3 = calculate_cagr(group, 3)
    cagr_5 = calculate_cagr(group, 5)

    cagr_results.append({
        "Scheme Name": scheme,
        "1Y CAGR": cagr_1,
        "3Y CAGR": cagr_3,
        "5Y CAGR": cagr_5
    })

In [ ]:
# Convert the List into a DataFrame
cagr_df = pd.DataFrame(cagr_results)
cagr_df.head()
cagr_display = cagr_df.copy()

cagr_display["1Y CAGR"] = cagr_display["1Y CAGR"] * 100
cagr_display["3Y CAGR"] = cagr_display["3Y CAGR"] * 100
cagr_display["5Y CAGR"] = cagr_display["5Y CAGR"] * 100

cagr_display.head()
cagr_df.to_csv(
    "../data/processed/cagr_comparison.csv",
    index=False
)

In [ ]:
# Sharpe Ratio — (Rp − Rf) / Std(Rp) × √252. Use Rf = 6.5% (RBI repo rate proxy). Rank all 40 funds.
# Compute Sharpe Ratio for all 40 funds

# Formula:
# Sharpe=Rp-Rf/σp x √252  

# Where

# Rp = Average Daily Return
# Rf = Risk Free Rate (6.5%)
# σ = Standard Deviation of Daily Returns
# √252 = Annualization factor
# Sort data

# 1. Import libraries
import pandas as pd
import numpy as np
# 2. Load data
nav = pd.read_csv("../data/processed/02_nav_history.csv")
# 3. Convert date
nav["date"] = pd.to_datetime(nav["date"])
# 4. Sort data
nav = nav.sort_values(["amfi_code", "date"])

# 5. Calculate daily_return
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)
# 6. Merge fund_master
fund = pd.read_csv("../data/processed/01_fund_master.csv")

fund.head()
fund.columns.tolist()
fund.info()
fund.columns

nav = nav.merge(
    fund[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "category",
            "expense_ratio_pct",
            "benchmark"
        ]
    ],
    on="amfi_code",
    how="left"
)
nav.head()
print(nav.columns)
rf = 0.065
daily_rf = rf/252
def sharpe_ratio(returns):

    excess = returns-daily_rf

    return (
        excess.mean()/returns.std()
    )*np.sqrt(252)


# Test on One Fund
import pandas as pd


first_scheme = nav["amfi_code"].unique()[0]

fund = nav[
    nav["amfi_code"] == first_scheme
]


def calculate_sharpe(returns, risk_free_rate=0.065):

    returns = returns.dropna()

    daily_rf = risk_free_rate / 252

    excess_return = returns - daily_rf

    mean_excess = excess_return.mean()

    std_return = returns.std()

    if std_return == 0:
        return np.nan

    sharpe = (mean_excess / std_return) * np.sqrt(252)

    return sharpe
sharpe_results = []
# Loop Through All 40 Funds
for scheme, group in nav.groupby("scheme_name"):

    sharpe = calculate_sharpe(group["daily_return"])

    sharpe_results.append({
        "Scheme Name": scheme,
        "Sharpe Ratio": sharpe
    })

    sharpe_df = pd.DataFrame(sharpe_results)
    # Rank the Funds
    sharpe_df = sharpe_df.sort_values(
    by="Sharpe Ratio",
    ascending=False
)

sharpe_df["Rank"] = range(1, len(sharpe_df) + 1)
sharpe_df.head(10)
sharpe_results

# save the results to a CSV file
sharpe_df.to_csv(
    "../data/processed/sharpe_ratio.csv",
    index=False
)

In [ ]:
# Sortino Ratio — same formula but denominator uses only downside standard deviation (negative return days only).
# Formula
# Sortino=Rp−Rf/Downside Standard Deviation × √252
# Notice:

# The numerator is exactly the same as Sharpe.

# Only the denominator changes.
def calculate_sortino(returns, risk_free_rate=0.065):

    # Remove missing values
    returns = returns.dropna()

    # Daily risk-free rate
    daily_rf = risk_free_rate / 252

    # Excess returns
    excess_returns = returns - daily_rf

    # Keep only negative returns
    downside_returns = returns[returns < 0]

    # Downside risk
    downside_std = downside_returns.std()

    # Avoid division by zero
    if downside_std == 0 or np.isnan(downside_std):
        return np.nan

    # Sortino Ratio
    sortino = (excess_returns.mean() / downside_std) * np.sqrt(252)

    return sortino


# Test on One Fund  like Sharpe.
import pandas as pd
import numpy as np
nav = pd.read_csv("../data/processed/02_nav_history.csv")

first_scheme = nav["amfi_code"].unique()[0]

fund = nav[
    nav["amfi_code"] == first_scheme
]

nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
sortino_results = []
# Merge fund_master to get scheme_name
fund = pd.read_csv("../data/processed/01_fund_master.csv")

nav = nav.merge(
    fund[
        [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "category",
            "expense_ratio_pct",
            "benchmark"
        ]
    ],
    on="amfi_code",
    how="left"
)

for scheme, group in nav.groupby("scheme_name"):

    ratio = calculate_sortino(group["daily_return"])

    sortino_results.append({
        "Scheme Name": scheme,
        "Sortino Ratio": ratio
    })

sortino_df = pd.DataFrame(sortino_results)
sortino_df = sortino_df.sort_values(
    by="Sortino Ratio",
    ascending=False
)
sortino_df["Rank"] = range(1, len(sortino_df) + 1)
sortino_df.head(10)
sortino_df.to_csv(
    "../data/processed/sortino_ratio.csv",
    index=False
)
    

In [ ]:
# Alpha and Beta — OLS regression of fund returns on Nifty 100 returns using scipy.stats.linregress. Alpha = intercept × 252.

import pandas as pd
import numpy as np
from scipy.stats import linregress

benchmark = pd.read_csv("../data/processed/10_benchmark_indices.csv")
nav = pd.read_csv("../data/processed/02_nav_history.csv")
benchmark["date"] = pd.to_datetime(benchmark["date"])
nav["date"] = pd.to_datetime(nav["date"])

print(benchmark.head())

print(benchmark.columns)
print(benchmark["index_name"].unique())

nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY 100"
].copy()

print(nifty100.head())

print(nifty100.shape)

# Calculate Benchmark Daily Returns
nifty100 = nifty100.sort_values("date")

nifty100["benchmark_return"] = (
    nifty100["close_value"].pct_change()
)

print(nifty100.head())

# Prepare the Fund NAV Data
print(nav.columns.tolist())

# daily return is already calculated in previous steps, but let's ensure it's there
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)
# If scheme_name is missing, we need to merge with fund_master to get scheme_name
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")

nav = nav.merge(
    fund_master[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

print(nav.shape)
print(nifty100.shape)
print(nav["date"].min(), nav["date"].max())
print(nifty100["date"].min(), nifty100["date"].max())


print(
    nav["date"].duplicated().sum()
)

# Merge Again
merged = pd.merge(
    nav,
    nifty100[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

print(merged.shape)
# Check the merged columns
print(merged.columns.tolist())

# Test Alpha & Beta on One Fund
from scipy.stats import linregress

first_scheme = merged["scheme_name"].iloc[0]

fund = merged[
    merged["scheme_name"] == first_scheme
].copy()

print(first_scheme)
print(fund.shape)

# Remove Missing Values
fund = fund.dropna(
    subset=["daily_return", "benchmark_return"]
)

print(fund.shape)

# Run Regression
result = linregress(
    fund["benchmark_return"],
    fund["daily_return"]
)

print(result)

# Calculate Alpha & Beta
alpha = result.intercept * 252
beta = result.slope

print("Alpha:", alpha)
print("Beta:", beta)

# Create a Function
def calculate_alpha_beta(group):

    group = group.dropna(
        subset=["daily_return", "benchmark_return"]
    )

    if len(group) < 2:
        return np.nan, np.nan

    result = linregress(
        group["benchmark_return"],
        group["daily_return"]
    )

    alpha = result.intercept * 252
    beta = result.slope

    return alpha, beta

# Calculate for All 40 Funds
alpha_beta_results = []

for scheme, group in merged.groupby("scheme_name"):

    alpha, beta = calculate_alpha_beta(group)

    alpha_beta_results.append({
        "Scheme Name": scheme,
        "Alpha": alpha,
        "Beta": beta
    })
# Create the Final DataFrame
    alpha_beta_df = pd.DataFrame(alpha_beta_results)

alpha_beta_df.head()

# Rank by Alpha
alpha_beta_df = alpha_beta_df.sort_values(
    by="Alpha",
    ascending=False
)
alpha_beta_df["Alpha Rank"] = (
    alpha_beta_df["Alpha"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Save the File
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)
print(nifty100.shape)
print(nav.columns.tolist())

# Merge NAV and Benchmark
merged = pd.merge(
    nav,
    nifty100[["date", "benchmark_return"]],
    on="date",
    how="inner"
)
print(merged.head())
print(merged.shape)


    


In [ ]:

# continue Alpha and Beta 
import pandas as pd
import numpy as np
from scipy.stats import linregress

benchmark = pd.read_csv("../data/processed/10_benchmark_indices.csv")
nav = pd.read_csv("../data/processed/02_nav_history.csv")
benchmark["date"] = pd.to_datetime(benchmark["date"])
nav["date"] = pd.to_datetime(nav["date"])
print(nav.shape)
print(nifty100.shape)
print(merged.shape)
print(nav["date"].min(), nav["date"].max())
print(nifty100["date"].min(), nifty100["date"].max())


print(
    nav["date"].duplicated().sum()
)

# Merge Again
merged = pd.merge(
    nav,
    nifty100[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

print(merged.shape)
# Check the merged columns
print(merged.columns.tolist())

# Test Alpha & Beta on One Fund
from scipy.stats import linregress

first_scheme = merged["scheme_name"].iloc[0]

fund = merged[
    merged["scheme_name"] == first_scheme
].copy()

print(first_scheme)
print(fund.shape)

# Remove Missing Values
fund = fund.dropna(
    subset=["daily_return", "benchmark_return"]
)

print(fund.shape)

# Run Regression
result = linregress(
    fund["benchmark_return"],
    fund["daily_return"]
)

print(result)

# Calculate Alpha & Beta
alpha = result.intercept * 252
beta = result.slope

print("Alpha:", alpha)
print("Beta:", beta)

# Create a Function
def calculate_alpha_beta(group):

    group = group.dropna(
        subset=["daily_return", "benchmark_return"]
    )

    if len(group) < 2:
        return np.nan, np.nan

    result = linregress(
        group["benchmark_return"],
        group["daily_return"]
    )

    alpha = result.intercept * 252
    beta = result.slope

    return alpha, beta

# Calculate for All 40 Funds
alpha_beta_results = []

for scheme, group in merged.groupby("scheme_name"):

    alpha, beta = calculate_alpha_beta(group)

    alpha_beta_results.append({
        "Scheme Name": scheme,
        "Alpha": alpha,
        "Beta": beta
    })
# Create the Final DataFrame
    alpha_beta_df = pd.DataFrame(alpha_beta_results)

alpha_beta_df.head()

# Rank by Alpha
alpha_beta_df = alpha_beta_df.sort_values(
    by="Alpha",
    ascending=False
)
alpha_beta_df["Alpha Rank"] = (
    alpha_beta_df["Alpha"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Save the File
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)

In [ ]:
# Maximum Drawdown — min(NAV / running_max − 1) for each fund. Find worst drawdown date range.
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import plotly.express as px 
nav = pd.read_csv("../data/processed/02_nav_history.csv")
nav["date"] = pd.to_datetime(nav["date"])   

print(nav.head())
nav["date"] = pd.to_datetime(nav["date"])
# Check Your NAV Data
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)
# If scheme_name is missing, we need to merge with fund_master to get scheme_name
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")
# merge nav with fund_master to get scheme_name
nav = nav.merge(
    fund_master[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

print(nav.shape)
print(nav["date"].min(), nav["date"].max())
print(nav.head())
# Sort the Data by Scheme Name and Date
nav = nav.sort_values(
    ["scheme_name", "date"]
)
# Calculate Running Maximum NAV
nav["running_max"] = (
    nav
    .groupby("scheme_name")["nav"]
    .cummax()
)

# Verify Running Maximum
nav[
    ["scheme_name","date","nav","running_max"]
].head(20)

# Calculate Daily Drawdown
nav["drawdown"] = (
    nav["nav"] /
    nav["running_max"]
) - 1

# Check the Drawdown Values
nav[
    ["nav","running_max","drawdown"]
].head(20)

# Find Maximum Drawdown for One Fund
first_scheme = nav["scheme_name"].iloc[0]

fund = nav[
    nav["scheme_name"] == first_scheme
]
print(first_scheme)

print(fund["drawdown"].min())

# Find the Worst Drawdown Date
worst_row = fund.loc[
    fund["drawdown"].idxmin()
]

print(worst_row)



In [ ]:
# print(nav.head())
# print(nav.columns.tolist())
# print(fund["drawdown"].min())
# print(worst_row)
# # Find the Peak Date
# # Worst drawdown row
# worst_row = fund.loc[
#     fund["drawdown"].idxmin()
# ]

# # Lowest point date
# trough_date = worst_row["date"]

# print("Trough Date:", trough_date)
# lowest_running_max = worst_row["running_max"]

# print(lowest_running_max)

# peak_row = fund[
#     fund["nav"] == lowest_running_max
# ].iloc[0]

# print(peak_row)

# print(f"Peak Date        : {peak_row['date'].date()}")
# print(f"Trough Date      : {trough_date.date()}")
# print(f"Maximum Drawdown : {fund['drawdown'].min():.2%}")

# Now Calculate for All 40 Funds
def calculate_max_drawdown(group):

    # Sort by date
    group = group.sort_values("date").copy()

    # Running maximum
    group["running_max"] = group["nav"].cummax()

    # Drawdown
    group["drawdown"] = (
        group["nav"] /
        group["running_max"]
    ) - 1

    # Worst drawdown row
    worst_idx = group["drawdown"].idxmin()

    worst_row = group.loc[worst_idx]

    # Peak row
    peak_row = group[
        group["nav"] == worst_row["running_max"]
    ].iloc[0]

    return pd.Series({

        "Max_Drawdown": worst_row["drawdown"],

        "Peak_Date": peak_row["date"],

        "Trough_Date": worst_row["date"]

    })

# Apply to All Funds
max_dd_df = (
    nav
    .groupby("scheme_name")
    .apply(calculate_max_drawdown)
    .reset_index()
)

# Check Output
print(max_dd_df.head())

# save the results to a CSV file
max_dd_df.to_csv(
    "../data/processed/maximum_drawdown.csv",
    index=False
)


In [ ]:
# Fund Scorecard (0–100) — composite: 30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × expense ratio rank (inverse) + 10% × max DD rank (inverse).

import pandas as pd
import numpy as np

# # Load the All DataFrames
# # Load Alpha & Beta
alpha_beta_df = pd.read_csv("../data/processed/alpha_beta.csv")

print(alpha_beta_df.head())

# Load Maximum Drawdown
max_dd_df = pd.read_csv("../data/processed/maximum_drawdown.csv")

print(max_dd_df.head())

# Load CAGR
cagr_df = pd.read_csv("../data/processed/cagr_comparison.csv")

print(cagr_df.head())

# Load Sharpe Ratio
sharpe_df = pd.read_csv("../data/processed/sharpe_ratio.csv")

print(sharpe_df.head())

# Load Fund Master
fund_master = pd.read_csv("../data/processed/01_fund_master.csv")
print(fund_master.head())

# Verify the Columns
print("Alpha Columns:")
print(alpha_beta_df.columns.tolist())
print("Max DD Columns:")
print(max_dd_df.columns.tolist())
print("CAGR Columns:")  
print(cagr_df.columns.tolist())
print("Sharpe Columns:")
print(sharpe_df.columns.tolist())
print("Fund Master Columns:")
print(fund_master.columns.tolist()) 

# Create the Base Scorecard
scorecard = cagr_df.copy()

print(scorecard.head())
# Merge Sharpe Ratio

scorecard = scorecard.merge(
    sharpe_df[["Scheme Name", "Sharpe Ratio"]],
    on="Scheme Name",
    how="left"
)

print(scorecard.head())
# Merge Alpha
scorecard = scorecard.merge(
    alpha_beta_df[["Scheme Name", "Alpha"]],
    on="Scheme Name",
    how="left"
)

print(scorecard.head())

# Merge Maximum Drawdown
scorecard = scorecard.merge(
    max_dd_df[["scheme_name", "Max_Drawdown"]],
    left_on="Scheme Name",
    right_on="scheme_name",
    how="left"
)

scorecard = scorecard.drop(columns="scheme_name")

print(scorecard.head())

# Merge Expense Ratio
scorecard = scorecard.merge(
    fund_master[["scheme_name", "expense_ratio_pct"]],
    left_on="Scheme Name",
    right_on="scheme_name",
    how="left"
)

scorecard = scorecard.drop(columns="scheme_name")

print(scorecard.head())
print(scorecard.shape)
print(scorecard.columns.tolist())

scorecard.shape
scorecard.columns.tolist()
scorecard.isnull().sum()

# # Calculate the Final Fund Score
# scorecard["Fund_Score"] = (

#     0.30 * scorecard["Cagr_Score"]

#     + 0.25 * scorecard["Sharpe_Score"]

#     + 0.20 * scorecard["Alpha_Score"]

#     + 0.15 * scorecard["Expense_Score"]

#     + 0.10 * scorecard["Drawdown_Score"]

# )

print(scorecard.columns.tolist())


In [ ]:
# Create Ranking Columns
# 1. CAGR Rank
scorecard["CAGR_Rank"] = scorecard["3Y CAGR"].rank(
    ascending=False,
    method="dense"
)
scorecard[["Scheme Name","3Y CAGR","CAGR_Rank"]].head()

# 2. Sharpe Rank
scorecard["Sharpe_Rank"] = scorecard["Sharpe Ratio"].rank(
    ascending=False,
    method="dense"
)
scorecard[["Scheme Name","Sharpe Ratio","Sharpe_Rank"]].head()

# 3. Alpha Rank
scorecard["Alpha_Rank"] = scorecard["Alpha"].rank(
    ascending=False,
    method="dense"
)
scorecard[["Scheme Name","Alpha","Alpha_Rank"]].head()

# 4. Expense Ratio Rank (Inverse)
scorecard["Expense_Rank"] = scorecard["expense_ratio_pct"].rank(
    ascending=True,
    method="dense"
)
scorecard[["Scheme Name","expense_ratio_pct","Expense_Rank"]].head()

# 5. Maximum Drawdown Rank (Inverse)
scorecard["Drawdown_Rank"] = scorecard["Max_Drawdown"].rank(
    ascending=False,
    method="dense"
)
scorecard[["Scheme Name","Max_Drawdown","Drawdown_Rank"]].head()

print(scorecard.columns.tolist())

# Convert Ranks to Scores (0–100)
# 1. CAGR Score
max_rank = scorecard["CAGR_Rank"].max()

scorecard["CAGR_Score"] = (
    (max_rank - scorecard["CAGR_Rank"]) /
    (max_rank - 1)
) * 100

# 2. Sharpe Score
max_rank = scorecard["Sharpe_Rank"].max()

scorecard["Sharpe_Score"] = (
    (max_rank - scorecard["Sharpe_Rank"]) /
    (max_rank - 1)
) * 100

# 3. Alpha Score
max_rank = scorecard["Alpha_Rank"].max()

scorecard["Alpha_Score"] = (
    (max_rank - scorecard["Alpha_Rank"]) /
    (max_rank - 1)
) * 100

# 4. Expense Ratio Score
max_rank = scorecard["Expense_Rank"].max()

scorecard["Expense_Score"] = (
    (max_rank - scorecard["Expense_Rank"]) /
    (max_rank - 1)
) * 100

# 5. Maximum Drawdown Score
max_rank = scorecard["Drawdown_Rank"].max()

scorecard["Drawdown_Score"] = (
    (max_rank - scorecard["Drawdown_Rank"]) /
    (max_rank - 1)
) * 100

# # Verify the Score Columns
print(scorecard.columns.tolist())

scorecard[
    [
        "Scheme Name",
        "CAGR_Score",
        "Sharpe_Score",
        "Alpha_Score",
        "Expense_Score",
        "Drawdown_Score"
    ]
].head()

# Calculate the Final Fund Score
scorecard["Fund_Score"] = (

    0.30 * scorecard["CAGR_Score"]

    + 0.25 * scorecard["Sharpe_Score"]

    + 0.20 * scorecard["Alpha_Score"]

    + 0.15 * scorecard["Expense_Score"]

    + 0.10 * scorecard["Drawdown_Score"]

)
scorecard["Fund_Score"] = scorecard["Fund_Score"].round(2)
# Create Overall Rank
scorecard["Overall_Rank"] = (
    scorecard["Fund_Score"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)
# Sort the Leaderboard
scorecard = scorecard.sort_values(
    by="Fund_Score",
    ascending=False
)
# View the Top 10 Funds
scorecard[
    [
        "Overall_Rank",
        "Scheme Name",
        "Fund_Score"
    ]
].head(10)

# Save the Deliverable
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print("Fund Scorecard saved successfully!")

# review the final scorecard
print(scorecard.shape)
# Check for missing values
scorecard[
    [
        "Overall_Rank",
        "Fund_Score"
    ]
].isnull().sum()



In [ ]:
# Benchmark comparison chart — plot top 5 funds vs Nifty 50 and Nifty 100 over 3 years. Compute tracking error = std(fund_return − benchmark_return) × √252.
# Tracking Error=Std(FundReturn−BenchmarkReturn)× 252
# Load All Required Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
nav = pd.read_csv("../data/processed/02_nav_history.csv")

benchmark = pd.read_csv("../data/processed/10_benchmark_indices.csv")

scorecard = pd.read_csv("../data/processed/fund_scorecard.csv")
# Convert Date
nav["date"] = pd.to_datetime(nav["date"])

benchmark["date"] = pd.to_datetime(benchmark["date"])

fund_master = pd.read_csv("../data/processed/01_fund_master.csv")
# Merge Scheme Names
nav = nav.merge(

    fund_master[["amfi_code","scheme_name"]],

    on="amfi_code",

    how="left"

)
nav.head()
# Find Top 5 Funds
top5 = scorecard.nsmallest(
    5,
    "Overall_Rank"
)

# print(top5[["Overall_Rank","Scheme Name"]])
# print(top5)
# Filter NAV for Top 5 Funds

top5_names = top5["Scheme Name"].tolist()

print(top5_names)

# Filter NAV History
top5_nav = nav[
    nav["scheme_name"].isin(top5_names)
].copy()

print(top5_nav.shape)
print(top5_nav["scheme_name"].unique())

# Keep Only the Last 3 Years
top5_nav = top5_nav[
    top5_nav["date"] >= "2023-01-01"
]
print(top5_nav["date"].min())
print(top5_nav["date"].max())

# Extract Nifty 50 and Nifty 100
print(benchmark["index_name"].unique())
# Filter Benchmark for Nifty 50 and Nifty 100
benchmark = benchmark[
    benchmark["index_name"].isin(["NIFTY50", "NIFTY100"])
].copy()

print(benchmark["index_name"].unique())
print(benchmark.shape)

# Keep Only the Last 3 Years
benchmark = benchmark[
    benchmark["date"] >= "2023-01-01"
]
print(benchmark["date"].min())
print(benchmark["date"].max())

# Normalize the Top 5 Funds
top5_nav = top5_nav.sort_values(
    ["scheme_name", "date"]
)

top5_nav["Normalized_NAV"] = (
    top5_nav.groupby("scheme_name")["nav"]
            .transform(lambda x: x / x.iloc[0] * 100)
)
top5_nav.head()
# Normalize NIFTY50 & NIFTY100
benchmark = benchmark.sort_values(
    ["index_name", "date"]
)

benchmark["Normalized_Index"] = (
    benchmark.groupby("index_name")["close_value"]
             .transform(lambda x: x / x.iloc[0] * 100)
)
benchmark.head()

# Plot the Comparison Chart
plt.figure(figsize=(14, 8))

# Plot Top 5 Funds
for scheme in top5_names:

    fund = top5_nav[
        top5_nav["scheme_name"] == scheme
    ]

    plt.plot(
        fund["date"],
        fund["Normalized_NAV"],
        label=scheme
    )

# Plot Benchmarks
for index in ["NIFTY50", "NIFTY100"]:

    bench = benchmark[
        benchmark["index_name"] == index
    ]

    plt.plot(
        bench["date"],
        bench["Normalized_Index"],
        linewidth=3,
        linestyle="--",
        label=index
    )

plt.title("Top 5 Funds vs NIFTY50 & NIFTY100 (Normalized)")
plt.xlabel("Date")
plt.ylabel("Normalized Value (Start = 100)")
plt.legend(loc="best", fontsize=8)
plt.grid(True)

plt.tight_layout()

plt.show()
# Save the Chart
plt.figure(figsize=(14, 8))

# Plot Top 5 Funds
for scheme in top5_names:
    fund = top5_nav[top5_nav["scheme_name"] == scheme]
    plt.plot(fund["date"], fund["Normalized_NAV"], label=scheme)

# Plot Benchmarks
for index in ["NIFTY50", "NIFTY100"]:
    bench = benchmark[benchmark["index_name"] == index]
    plt.plot(
        bench["date"],
        bench["Normalized_Index"],
        linewidth=3,
        linestyle="--",
        label=index
    )

plt.title("Top 5 Funds vs NIFTY50 & NIFTY100")
plt.xlabel("Date")
plt.ylabel("Normalized Value (Start = 100)")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    "../reports/benchmark_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Tracking Error=Std(FundReturn−BenchmarkReturn)×252​

# Calculate Benchmark Daily Returns
benchmark = benchmark.sort_values(
    ["index_name", "date"]
)

benchmark["benchmark_return"] = (
    benchmark
    .groupby("index_name")["close_value"]
    .pct_change()
)

benchmark.head()
# Choose NIFTY100
nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

print(nifty100.head())

# Merge Fund Returns with NIFTY100 Returns
top5_nav = top5_nav.sort_values(
    ["scheme_name", "date"]
)

top5_nav["fund_return"] = (
    top5_nav
    .groupby("scheme_name")["nav"]
    .pct_change()
)

tracking = pd.merge(

    top5_nav,

    nifty100[
        ["date", "benchmark_return"]
    ],

    on="date",

    how="inner"

)
# Verify Merge
print(tracking.shape)

tracking.head()

# Create Tracking Error Function
import numpy as np

def tracking_error(group):

    difference = (
        group["fund_return"]
        -
        group["benchmark_return"]
    )

    return difference.std() * np.sqrt(252)
# Calculate for All Top 5 Funds
tracking_results = []

for scheme, group in tracking.groupby("scheme_name"):

    te = tracking_error(group)

    tracking_results.append({

        "Scheme Name": scheme,

        "Tracking Error": te

    })
    # Create DataFrame
    tracking_df = pd.DataFrame(
    tracking_results
)

tracking_df
# Rank Tracking Error
# Lower Tracking Error = Better.

tracking_df = tracking_df.sort_values(
    by="Tracking Error"
)

tracking_df["Rank"] = range(
    1,
    len(tracking_df)+1
)

tracking_df

# Save the File
tracking_df.to_csv(
    "../data/processed/tracking_error.csv",
    index=False
)

print("Tracking Error file saved successfully!")

# Verify the Output
print(tracking_df)